In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
from fundus_data_toolkit.functional import open_image
from jppype import Mosaic, vscode_theme

from fundus_odmac_toolkit.models.segmentation import segment
from fundus_toolkits import FundusData
from fundus_vessels_toolkit import VTree
from fundus_vessels_toolkit.models import segment_av
from fundus_vessels_toolkit.pipelines.avseg_to_tree import GNNAVSegToTree, NaiveAVSegToTree
from fundus_vessels_toolkit.segment_to_graph.tree_topology import TreeTopology, optimal_lines
from fundus_vessels_toolkit.segment_to_graph.vbranch_digraph import VBranchDigraph
from fundus_vessels_toolkit.utils.jppype import draw_graph, draw_tree, draw_trees

vscode_theme()

HTML(value="<style>\n        .cell-output-ipywidget-background {\n                background: transparent !imp…

HTML(value="<style>\n        .cell-output-ipywidget-background {\n                background: transparent !imp…

In [3]:
from fundus_vessels_toolkit.segment_to_graph.tree_topology import TopologicalLabel


def draw_topology(topo: TreeTopology, img=None):
    if img is None:
        img = fundus.image.transpose(1, 2, 0) * 0.5
    color_map = np.zeros(topo.shape + (3,), dtype=np.float32)
    alpha = np.zeros(topo.shape, dtype=np.float32)

    subtree_map = TopologicalLabel.decode_subtree(topo.branch_map)
    N_subtree = int(subtree_map.max()) + 1

    for s in range(0, N_subtree):
        mask = subtree_map == s
        if mask.sum() == 0:
            continue
        color_map[mask] = TopologicalLabel.subtree_color(s, format="rgb") / 255.0
        subtree_topo = topo.rank_map[mask]
        alpha[mask] = 1 - 0.8 * (np.floor(subtree_topo) + subtree_topo) / (subtree_topo.max() * 2)

    alpha = alpha[:, :, None]

    return (1 - alpha) * img + alpha * color_map


def draw_topos(topos):
    img = fundus.image.transpose(1, 2, 0) * 0.5
    img = draw_topology(topos[0], img)
    img = draw_topology(topos[1], img)
    return img

## Load Image and Segment AV, OD, Macula


In [ ]:
PATH = Path("/run/media/gaby/GREY SSD/PostDoc/DATA/Fundus/Fundus-AV/")
RAW = PATH / "1-images"
AV = PATH / "2-av"
TOPO = PATH / "3-topo"
IMG = sorted(list(AV.glob("*.png")))[10].stem  # 18
# IMG = "015_N"

fundus_gt = FundusData(image=RAW / (IMG + ".png"), av=AV / (IMG + ".png"))
trees_gt = VTree.load(TOPO / f"{IMG}_art.npz"), VTree.load(TOPO / f"{IMG}_vei.npz")
topo_gt = (
    TreeTopology.from_tree(trees_gt[0], expand_labels_by=10, sparse=False, discard_tree=True),
    TreeTopology.from_tree(trees_gt[1], expand_labels_by=10, sparse=False, discard_tree=True),
)
sparse_topo_gt = (
    TreeTopology.from_tree(trees_gt[0], expand_labels_by=10, sparse=True, discard_tree=True),
    TreeTopology.from_tree(trees_gt[1], expand_labels_by=10, sparse=True, discard_tree=True),
)

# od_mac = segment(open_image(RAW / (IMG + ".png"))).numpy(force=True).argmax(axis=0)
# fundus_gt = fundus_gt.update(od=od_mac == 1, macula=od_mac == 2, reshape_method="resize")
fundus = fundus_gt.copy()
_ = segment_av(fundus)

av2tree = GNNAVSegToTree()
graph = av2tree.to_vgraph(fundus).sort_branches_by_nodesID()

print(IMG)

011_N


In [ ]:
graph = av2tree.to_vgraph(fundus)

In [6]:
f"{(sparse_topo_gt[0].__sizeof__() + sparse_topo_gt[1].__sizeof__()) / 8 * 1e-6:.2} MB vs {(topo_gt[0].__sizeof__() + topo_gt[1].__sizeof__()) / 8 * 1e-6:.2} MB"

'7.2 MB vs 1.9e+01 MB'

In [7]:
digraph = VBranchDigraph.from_graph(graph, max_distance=200, max_angle=45)
line_p = digraph.compute_p_from_gt(topo_gt[0], topo_gt[1])
digraph.compute_p_from_gt(sparse_topo_gt[0], sparse_topo_gt[1])
assert np.all(np.isclose(line_p, digraph.compute_p_from_gt(sparse_topo_gt[0], sparse_topo_gt[1]))), (
    "Line probabilities from sparse and dense topology should be identical."
)

solved_tree = digraph.optimize_tree(keep_invalid_branch=True)

m = Mosaic(
    3, cols_titles=["Predicted", "Predicted with GT Topology", "Ground Truth"], cell_height=700, background=fundus.image
)
fundus.draw(view=m[0])
draw_graph(digraph.graph, view=m[0], edge_labels=True, node_labels=True)
m[1].add_image(fundus.image, name="fundus")
# m[1].add_image(draw_topology(topo_gt[0]), name="topos")
# draw_graph(sol, view=m[1])
fundus.draw(view=m[1])
draw_tree(
    solved_tree,
    view=m[1],
    branch_color="subtree",
    bspline_dir=True,
)

fundus_gt.draw(view=m[2])
# m[2].add_image(draw_topology(topo_gt[0]), name="topo")
# m[2].add_image(
#     np.stack(
#         [
#             np.zeros_like(topo_gt[0].fuzzy_skeleton_map),
#             np.zeros_like(topo_gt[0].fuzzy_skeleton_map),
#             topo_gt[1].fuzzy_skeleton_map,
#         ]
#     ).transpose(1, 2, 0),
#     name="skeleton",
#     opacity=0.9,
# )
# m[2].add_image(
#     np.stack(
#         [
#             topo_gt[0].fuzzy_skeleton_map,
#             np.zeros_like(topo_gt[0].fuzzy_skeleton_map),
#             topo_gt[1].fuzzy_skeleton_map,
#         ]
#     ).transpose(1, 2, 0),
#     name="skeleton",
#     opacity=0.9,
# )
draw_trees(trees_gt, view=m[2], bspline_dir=True)  # , edge="skeleton")
m


[ WARN:0@9.643] global loadsave.cpp:1617 imencodeWithMetadata Unsupported depth image for selected encoder is fallbacked to CV_8U.


GridBox(children=(HTML(value='<h3 style="text-align: center;">Predicted</h3>'), HTML(value='<h3 style="text-al…

In [8]:
solved_tree.branch_tree[224]

np.int64(23)

In [9]:
solved_tree.branch_tree[298]

np.int64(195)

In [10]:
m.views[0].goto(digraph.graph.branch(6).midpoint()[::-1], 5)

In [11]:
digraph.compute_p_from_gt(topo_gt[0], topo_gt[1])

array([False, False, False, ..., False, False, False], shape=(6463,))

In [12]:
np.argwhere(line_p != digraph.compute_p_from_gt(sparse_topo_gt[0], sparse_topo_gt[1]))

array([], shape=(0, 1), dtype=int64)

In [13]:
m.views[0].goto(digraph.graph.branch(12).midpoint()[::-1], scale=4)

In [14]:
VBranchDigraph.from_graph(graph, max_distance=200, max_angle=45)

In [15]:
digraph.compute_p_from_gt(topo_gt[0], topo_gt[1])

array([False, False, False, ..., False, False, False], shape=(6463,))

In [ ]:
np.concatenate([tip_rank1, tip_rank2, np.arange(label1.shape[0])[:, None]], axis=1)[
    np.any(tip_rank1 != tip_rank2, axis=1)
]

NameError: name 'tip_rank1' is not defined

In [ ]:
from fundus_vessels_toolkit.segment_to_graph.tree_topology import read_branch_topology

for art in [1, 0]:
    label1, dir1, plau1, tip_label1, tip_rank1 = read_branch_topology(digraph.graph, topo_gt[art])
    label2, dir2, plau2, tip_label2, tip_rank2 = read_branch_topology(digraph.graph, sparse_topo_gt[art])
    assert np.all(label1 == label2), "Branch labels from sparse and dense topology should be identical."
    assert np.all(np.isclose(dir1, dir2)), "Branch directions from sparse and dense topology should be identical."
    assert np.all(tip_label1 == tip_label2), "Tip labels from sparse and dense topology should be identical."
    assert np.all(np.isclose(tip_rank1, tip_rank2)), "Tip ranks from sparse and dense topology should be identical."
    assert np.all(np.isclose(plau1, plau2)), "Branch plausibility from sparse and dense topology should be identical."


AssertionError: Tip ranks from sparse and dense topology should be identical.

In [ ]:
%timeit VBranchDigraph.from_graph(graph, max_distance=200)
%timeit digraph.compute_p_from_gt(sparse_topo_gt[0], sparse_topo_gt[1])
%timeit digraph.optimize_tree(keep_invalid_branch=True)

41.4 ms ± 857 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)
15.2 ms ± 375 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
101 ms ± 2.8 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
